# **Processamento de Linguagem Natural [2024-Q2]**
Prof. Alexandre Donizeti Alves

### **PROJETO PRÁTICO** [LangChain + Grandes Modelos de Linguagem + API]


*Course project: Natural Language Processing (UFABC, 2024-Q2).*


### **EQUIPE**

---

**Author:** Enzo Fujita Garcia


### **GRANDE MODELO DE LINGUAGEM (*Large Language Model - LLM*)**

---

Dados do LLM selecionada:

>


**LLM**: GPT-4o mini (Generative Pre-trained Transformer) da OpenAI

>

**Link para a documentação oficial**: https://platform.openai.com/docs/concepts


### **API**
---

*(Em nosso projeto utilizamos duas API's diferentes que se complementam para o nosso objetivo final)*

1:

**API**: Steam

**Site oficial**: https://pypi.org/project/python-steam-api/

**Link para a documentação oficial**: https://steamcommunity.com/dev

2:

**API**: Steam Reviews

**Site oficial**: https://pypi.org/project/steamreviews/

**Link para a documentação oficial**: https://github.com/woctezuma/steam-reviews

---

# **Descrição do Projeto:**

O projeto trata-se da utilização em conjunto de duas API's: **API da Steam** e **API Steam Reviews**, que são complementares, com ambas analisam dados da plataforma de distribuição de jogos digitais (games) para computadores, Steam.

O objetivo do projeto visa **analisar os dados da conta de um determinado usuário e estimar recomendações de games a partir de suas informações** como: games mais jogados pelo usuário, games mais jogados pelos amigos do usuário, quais gêneros de games o usuário mais joga, etc.

Para obter essas informações, foram utilizadas técnicas de **Processamento de Linguagem Natural (PLN)** para a execução de análises e geração de conteúdo com a ajuda de um **Grande Modelo de Linguagem (LLM)**, que para esse projeto foi o **GPT-4o mini**.


**Principais técnicas de PLN utilizadas no projeto:**

- Análise de Sentimentos

- Detecção de Emoções

- Extração de Palavras-chave

- Tradução de Textos

- Sumarização de Textos

---

##**Project walkthrough video:** *(add link here)*


# **Criação do Projeto:**


#**1) Realização das configurações primárias**

- **Instalando as API's que serão utilizadas**

In [ ]:
!pip install openai
!pip install python-steam-api
!pip install steamreviews

- **Introduzindo as chaves**

In [ ]:
from getpass import getpass
OPENAI_API_KEY = getpass()
from openai import OpenAI
import json

cliente = OpenAI(api_key= OPENAI_API_KEY)

In [ ]:
from getpass import getpass
STEAM_API_KEY = getpass("Steam API key: ")
import os
from steam_web_api import Steam
steam = Steam(STEAM_API_KEY)

#**2) Obtendo as informações do usuário**

***(É importante destacar que devido aos recursos de privação de detalhes da conta, alguns recursos do projeto podem não funcionar com determinados usuários pelo fato de eles utilizarem esse recurso para esconder suas informações)***

- **Inserindo o ID da conta do usuário**

In [ ]:
user_id = input("Digite o ID do usuário: ")

- **Lista de games do usuário**

In [ ]:
user_games = steam.users.get_owned_games(user_id)
user_games

- **Lista em ordem decrescente**

In [ ]:
sorted_games = sorted(user_games['games'], key=lambda x: x['playtime_forever'], reverse=True)

jogos = []
user_sorted_games = ""
# Exibindo os jogos ordenados com nome, playtime e appid
for game in sorted_games:
    jogo = (f"Jogo: {game['name']}, Playtime: {game['playtime_forever']} minutos, AppID: {game['appid']}")
    jogos.append(jogo)

print(jogos)

for i in jogos:
  user_sorted_games += i + "\n"

# Váriavel que armazena os nomes de cada jogo = user_sorted_games
print(user_sorted_games)

- **Lista de amigos do usuário**

In [ ]:
user_friends = steam.users.get_user_friends_list(user_id)
user_friends

- **Games mais jogados pelos amigos do usuário**

*(Note que não há repetições)*

In [ ]:
steam_ids = [friend['steamid'] for friend in user_friends['friends']]
lista_jogo_mais_jogado = []
for steam_id in steam_ids:
    amigos_id = steam.users.get_owned_games(steam_id)
    if len(amigos_id) == 0:
        continue
    sorted_games = sorted(amigos_id['games'], key=lambda x: x['playtime_forever'], reverse=True)
    jogo_mais_jogado = sorted_games[0]
    nome_jogo_mais_jogado = jogo_mais_jogado['name']
    lista_jogo_mais_jogado.append(nome_jogo_mais_jogado)

# Removendo duplicatas
lista_jogo_mais_jogado = list(dict.fromkeys(lista_jogo_mais_jogado))

# Variável de texto com todos os nomes dos jogos mais jogados
user_friends_most_played_games = "\n" + '\n'.join([f"- {nome}" for nome in lista_jogo_mais_jogado])

# Exibindo o texto
print(user_friends_most_played_games)

#**3) Utilizando o LLM para obter recomendações de games**

- **Gerando uma lista de recomendações a partir dos games mais jogados pelo usuário**

In [ ]:
resposta = cliente.chat.completions.create(
    messages=[
        {"role": "system", "content": "You'll receive a list of games, and you must recommend other 10 games based on this list. You can't recommend games that are on the list. Answer in brazillian portuguese"},
        {"role": "user", "content": f"{user_sorted_games}"}
    ],
    model="gpt-4o-mini"
)
dados = json.loads(resposta.json())
json_formatado = json.dumps(dados, indent=4, ensure_ascii=False)
linhas = json_formatado.splitlines()
print(resposta.choices[0].message.content)

##Técnica 1: Sumarização

- **Resumo dos gostos pessoais do usuário a partir de seus games mais jogados**

In [ ]:
resposta = cliente.chat.completions.create(
    messages=[
        {"role": "system", "content": "You are an summarizer. You'll receive a list of games. Based on that, you must summarize the game taste of the user. Answer in portuguese."},
        {"role": "user", "content": f"{user_sorted_games}"}
    ],
    model="gpt-4o-mini"
)
dados = json.loads(resposta.json())
json_formatado = json.dumps(dados, indent=4, ensure_ascii=False)
linhas = json_formatado.splitlines()
resposta.choices[0].message.content

- **Resumo dos gostos pessoais dos amigos do usuário a partir do game mais jogado por cada um**

In [ ]:
resposta = cliente.chat.completions.create(
    messages=[
        {"role": "system", "content": "You are an summarizer. You'll receive a list of games. Based on that, you must summarize the game taste of my friends. Answer in portuguese."},
        {"role": "user", "content": f"{user_friends_most_played_games}"}
    ],
    model="gpt-4o-mini"
)
dados = json.loads(resposta.json())
json_formatado = json.dumps(dados, indent=4, ensure_ascii=False)
linhas = json_formatado.splitlines()
resposta.choices[0].message.content

- **Criação de uma gráfico de barras a partir do tempo gasto pelo usuário em cada game**

In [ ]:
import matplotlib.pyplot as plt
jogo = []
tempos_jogados = []

# Extração dos dados de cada string
for dado in jogos:
    if 'Playtime:' in dado:
        # Separando o nome do jogo
        nome_jogo = dado.split(',')[0].replace('Jogo: ', '').strip()
        jogo.append(nome_jogo)

        # Extraindo o tempo jogado
        playtime = int(dado.split('Playtime: ')[1].split(' ')[0])
        tempos_jogados.append(playtime)


# Exibindo os resultados
print("Jogos:", jogo)
print("Tempos Jogados:", tempos_jogados)


# Converte minutos para horas para uma visualização mais compreensível
tempos_jogados_horas = [tempo / 60 for tempo in tempos_jogados]


# Gráfico de barras horizontal
plt.figure(figsize=(28,5))
plt.bar(jogo[:10], tempos_jogados_horas[:10], color='gray')

# Adicionando títulos e rótulos
plt.xlabel('Tempo Jogado (em horas)')
plt.ylabel('Jogos')
plt.title('Tempo Jogado por Jogo (em horas)')

# Exibindo o gráfico
plt.tight_layout()
plt.show()

#**4) Análise de gêneros**

##Técnica 2: Extração de Palavras-Chave

- **Extraindo os ID's dos games mais jogados pelo usuário**

*(Os ID's estão na mesma ordem do gráfico, ou seja, estão em ordem descrescente de tempo de jogo)*

In [ ]:
def extrair_ids(lista_de_jogos):
  app_ids = []

  for dado in jogos:
      partes = dado.split('AppID: ')
      if len(partes) > 1:
          app_id = partes[1].split()[0].strip()
          app_ids.append(int(app_id))

  return app_ids

app_ids = extrair_ids(jogos)
print(app_ids)

- **Teste para a extração de gêneros de um game**

In [ ]:
def extrair_descriptions(json_data):
    descriptions = []

    # Itera sobre cada jogo no dicionário
    for app_id, details in json_data.items():
        if details['success']:
            genres = details.get('data', {}).get('genres', [])
            # Adiciona a descrição de cada gênero à lista
            descriptions.extend([genre['description'] for genre in genres])

    return descriptions

# TESTANDO
resposta_json = steam.apps.get_app_details(app_ids[0] , filters = "genres")
descriptions = extrair_descriptions(resposta_json)
print(descriptions)

- **Extração de gêneros de cada game jogado pelo usuário**

In [ ]:
def generos_jogo(app_ids):
  generos = []
  for i in range(3):
    resposta_json = steam.apps.get_app_details(app_ids[i] , filters = "genres")
    descriptions = extrair_descriptions(resposta_json)
    generos.append(descriptions)
  return generos

generos = generos_jogo(app_ids)

- **Identificação dos games com seu respectivo gênero**

In [ ]:
sorted_games = sorted(user_games['games'], key=lambda x: x['playtime_forever'], reverse=True)
jogos = []
user_sorted_games = ""

for game in sorted_games:
    jogo = (f"Jogo: {game['name']}")
    jogos.append(jogo)

for i in jogos:
  user_sorted_games += i + "\n"


variavel = ""
for i in range(len(generos)):
  variavel += f"{jogos[i]} Gêneros: {(generos[i])}  \n"

print(variavel)

## Técnica 3: Tradução de textos

- **Tradução dos gêneros de jogos do inglês para o português e identificação da predominância percentual de gêneros**

In [ ]:
resposta = cliente.chat.completions.create(
    messages=[
        {"role": "system", "content": "You are an great assistant.Translate the genre in brazillian portuguese and provide the predominance in % of each one based on the file."},
        {"role": "user", "content": f"{variavel}. Describe the favorites genres of the player"}
    ],
    model="gpt-4o-mini"
)
dados = json.loads(resposta.json())
json_formatado = json.dumps(dados, indent=4, ensure_ascii=False)
linhas = json_formatado.splitlines()
print(resposta.choices[0].message.content)

- **Extração de gêneros dos games mais jogados pelos amigos do usuário**

In [ ]:
resposta = cliente.chat.completions.create(
    messages=[
        {"role": "system", "content": "You are an great assistant. You'll receive a list of games. Based on that, extract key-words (e.g. the genres). Answer in brazillian portuguese"},
        {"role": "user", "content": f"{user_friends_most_played_games}. Just tell me the key-words, and nothing else."}
    ],
    model="gpt-4o-mini"
)
dados = json.loads(resposta.json())
json_formatado = json.dumps(dados, indent=4, ensure_ascii=False)
linhas = json_formatado.splitlines()
print(resposta.choices[0].message.content)

#**5) Reviews de um determinado jogo realizado por outros usuários da Steam**

- **Selecionando o game de interesse**

In [ ]:
import requests

# Define o ID do aplicativo
appid = input("Digite o ID do game: ")

- **Buscando críticas sobre o jogo**

In [ ]:
# Define a URL da API
url = f"https://store.steampowered.com/appreviews/{appid}?json=1"

# Faz a requisição GET
response = requests.get(url)

# Verifica se a requisição foi bem-sucedida
if response.status_code == 200:
    # Converte a resposta para um objeto JSON
    data = response.json()

    # Verifica se o parâmetro 'reviews' está presente na resposta
    if 'reviews' in data:
        # Extrai apenas os textos das reviews
        reviews = [review['review'] for review in data['reviews']]

        # Exibe ou processa as reviews conforme necessário
        for review in reviews:
            print(review)
    else:
        print("Parâmetro 'reviews' não encontrado na resposta.")
else:
    print(f"Falha na requisição. Código de status: {response.status_code}")


##Técnicas 4: Detecção de emoções

- **Detecção de emoções nas críticas**

In [ ]:
resposta = cliente.chat.completions.create(
    messages=[
        {"role": "system", "content": "You're a great assistant. You'll receive a list of reviews and you must classify the general feelings of the reviews"},
        {"role": "user", "content": f"{reviews}. Answer in brazillian portuguese"}
    ],
    model="gpt-4o-mini"
)
dados = json.loads(resposta.json())
json_formatado = json.dumps(dados, indent=4, ensure_ascii=False)
linhas = json_formatado.splitlines()
print(resposta.choices[0].message.content)

##Técnicas 5: Análise de sentimentos

- **Análise dos sentimentos dos usuários segundo as caracteríticas do jogo**

In [ ]:
resposta = cliente.chat.completions.create(
    messages=[
        {"role": "system", "content": "You're a great assistant. You'll receive a list of reviews and you must do a summary of the general feelings on the reviews in topics, for example story, soundtrack etc"},
        {"role": "user", "content": f"{reviews}. Answer in brazillian portuguese"}
    ],
    model="gpt-4o-mini"
)
dados = json.loads(resposta.json())
json_formatado = json.dumps(dados, indent=4, ensure_ascii=False)
linhas = json_formatado.splitlines()
print(resposta.choices[0].message.content)